# Prompt Iteration — Pairwise (A/B) Evaluation & Align Evals for the Support Agent

This module iterates on the **voice + chat customer support agent** for a smart-TV brand
(`src/pov_agent/agent.py`) — the same agent as the [LangSmith walkthrough](./langsmith_walkthrough.ipynb),
but this notebook is **self-contained**: the *Standalone setup* section below reproduces every
shared symbol it needs (`client`, `agent`, `run_support_agent`, `target`, `judge_model`,
`dataset_name`, `scoped`, `PROJECT`, `create_run_rule`, `ensure_queue`, `queue`, `voice_cases`),
so you can run it top-to-bottom on a fresh kernel without opening the walkthrough first. If you
*have* already run the walkthrough in this kernel, the setup cells are idempotent — they'll just
rebind the same names.

The agent serves two channels from one graph: **voice** (it receives a speech-to-text transcript
and its reply is spoken by text-to-speech) and **chat**. It handles data shaped like smart-TV
**ACR viewer data** — device and household identifiers, IP/MAC addresses, viewing history — plus
ordinary account PII, behind three prevention layers:

| Level | Control | Where |
|---|---|---|
| 1 | **PII middleware** on input, output, *and* tool results | `privacy.build_pii_middleware()` |
| 2 | **LLM Gateway** redaction + spend limits | `config.get_model()` (`USE_LANGSMITH_GATEWAY`) |
| 3 | **LangSmith SDK masking** before trace egress | `privacy.install_trace_masking()` |

Where the walkthrough scores each experiment **absolutely**, this module adds two iteration tools:

1. **Pairwise (A/B) evaluation** — when you're choosing between two candidates (a cheaper or
   newer model vs. the incumbent), the question is *relative*: which support reply is better? An
   LLM judge sees both replies side by side and picks a winner.
2. **Align Evals** — tune a subjective, house-style judge (here, **`voice_ready`**) to agree with
   a human expert before you trust it in an online evaluator.

> Every experiment, judge call, and trace in this notebook goes through the masking client, so
> what LangSmith stores — and what the judges and human reviewers see — is **already redacted**.
> All customers, accounts, and identifiers are synthetic (`src/pov_agent/support_data.py`).

**Prerequisites:** a `.env` with a real `LANGSMITH_API_KEY`, `LANGSMITH_TRACING=true`,
`LANGSMITH_PROJECT=<your project>`, and either the LLM Gateway enabled (the default) or the
provider key your `MODEL` points at.

## Standalone setup

Everything this module depends on, reproduced from the walkthrough so the notebook runs in
isolation. Run these four cells once on a fresh kernel; skip them if the walkthrough already ran
in this kernel. The dataset is built from the repo's golden support turns
(`seed_data.SCENARIOS`) — six voice and six chat.

### Setup 1 — environment, masking client, and the agent

Load `.env`, make `pov_agent` importable from anywhere under the repo, and create the LangSmith
client with **`install_trace_masking()`** instead of a bare `Client()`. That returns a `Client`
with an anonymizer attached *and* installs it as the process-wide tracing client, so experiment
runs, judge calls, and agent traces are all scrubbed in-process before they leave the machine
(Level 3). `build_agent()` wires in the PII middleware (Level 1) and reads `MODEL` /
`USE_LANGSMITH_GATEWAY` via `get_model()` (Level 2).

In [ ]:
import os
import re
import sys
import time
import uuid
import logging
from pathlib import Path

# Make the pov_agent package importable whether the notebook runs from the repo
# root, from src/, or from src/pov_agent/modules/.
here = Path().resolve()
src_dir = None
for candidate in (here, *here.parents):
    if (candidate / "pov_agent" / "agent.py").exists():
        src_dir = candidate
        break
    if (candidate / "src" / "pov_agent" / "agent.py").exists():
        src_dir = candidate / "src"
        break
if src_dir is None:
    raise RuntimeError("Could not locate the pov_agent package on any parent path.")
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dotenv import load_dotenv
load_dotenv(dotenv_path=src_dir.parent / ".env", override=True)

from pov_agent.agent import build_agent
from pov_agent.config import get_model
from pov_agent.privacy import install_trace_masking

# Deep agents run subagents/tools on parallel threads; LangSmith's tracer logs
# benign "No indexed run ID" warnings for those. Filter ONLY that message.
class _NoIndexedRunIDFilter(logging.Filter):
    def filter(self, record):
        return "No indexed run ID" not in record.getMessage()

logging.getLogger("langchain_core.callbacks.manager").addFilter(_NoIndexedRunIDFilter())

# Level 3 — an anonymizing Client, installed as the process-wide tracing client.
client = install_trace_masking()
PROJECT = os.environ.get("LANGSMITH_PROJECT", "pov-support-agent")

# Build the agent once and reuse it. Its Context Hub memory is read-only, so
# experiments can run with modest concurrency.
agent = build_agent()

# Shared judge model — independent of the agent, read from the same MODEL env var
# (and routed through the LLM Gateway when that's enabled).
judge_model = get_model()

# Per-run resource scoping so re-runs don't collide with a neighbor's resources.
def scoped(name):
    who = (os.environ.get("WORKSHOP_USER") or "demo").lower().replace(" ", "-")
    return f"{name}-{who}"

print("MODEL:", os.environ.get("MODEL", "(default)"))
print("LLM Gateway:", os.environ.get("USE_LANGSMITH_GATEWAY", "true"))
print("Project:", PROJECT)
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))

### Setup 2 — drive & read the agent

A support turn is a flat dict — the same shape as a dataset example's inputs:

```python
{"channel": "voice", "account_id": "ACC-1001", "identity_verified": True,
 "message": "<customer message or STT transcript>"}
```

`run_support_agent` wraps the repo's `pov_agent.runner.run_turn`, which invokes the agent with
the session fields in graph state (tools read `account_id` from state, so the model never handles
it) and returns `{response, tools_called, actions}` — `actions` being the account-changing tools
it called (`set_viewing_data_collection`, `submit_privacy_request`, `create_support_ticket`).
`target` is the thin adapter the offline experiments call; `voice_cases` are the voice scenarios
the Align Evals section reuses.

In [ ]:
from pov_agent.runner import run_turn
from pov_agent.langsmith.seed_data import INPUT_KEYS, REFERENCE_KEYS, SCENARIOS

def run_support_agent(inputs, agent_=None, metadata=None, tags=None):
    """Run one support turn and return {response, tools_called, actions}."""
    return run_turn(
        agent_ or agent,
        inputs,
        metadata={"source": "notebook", **(metadata or {})},
        tags=["notebook", *(tags or [])],
    )

def target(inputs: dict) -> dict:
    return run_support_agent(inputs)

# Voice scenarios, reused by the Align Evals section.
voice_cases = [{k: s[k] for k in INPUT_KEYS} for s in SCENARIOS if s["channel"] == "voice"]
print(f"Agent ready. {len(voice_cases)} voice cases loaded.")

### Setup 3 — the dataset

The repo's golden support turns: inputs are `channel`, `account_id`, `identity_verified`, and
`message`; reference outputs are the `must_address` points, the per-turn `privacy_expectation`,
and the `expected_actions` / `forbidden_actions` — the fields the pairwise judge reads. Several
messages deliberately contain sensitive data in the customer's own words (an email, a phone number
read aloud, a card number, an SSN) or ask for data the agent must not disclose (viewing history,
an IP address, a teenager's activity).

Idempotent: it recreates `dataset_name` if it already exists. Dataset uploads are a direct write —
the anonymizer only scrubs *runs* — which is why the examples are synthetic.

In [ ]:
examples = [
    {
        "inputs": {k: s[k] for k in INPUT_KEYS},
        "outputs": {k: s[k] for k in REFERENCE_KEYS},
        "metadata": {"channel": s["channel"]},
    }
    for s in SCENARIOS
]
dataset_name = scoped("pov-support-iteration")

if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(
    dataset_name,
    description="Golden voice + chat support turns for a smart-TV brand (synthetic).",
)
client.create_examples(dataset_id=dataset.id, examples=examples)
n_voice = sum(1 for s in SCENARIOS if s["channel"] == "voice")
print(f"Created dataset '{dataset_name}' with {len(examples)} examples "
      f"({n_voice} voice + {len(examples) - n_voice} chat)")
print(f"View: {dataset.url}")

### Setup 4 — run rules & the annotation queue

The `create_run_rule` helper (same as the walkthrough's Part 3/4) and the support `queue` that
the Align Evals section routes voice replies into for human labeling. The rubric mirrors
`src/pov_agent/langsmith/annotation_queues.py` (`resolution_quality`, `privacy_compliant`,
`channel_appropriate`, `reviewer_notes`) — imported, so the notebook can't drift from the repo —
plus one extra pass/fail item, **`voice_ready`**, for the house-style label we align on below.
`ensure_queue` is idempotent — it reuses an existing queue by name.

In [ ]:
from pov_agent.langsmith.annotation_queues import _FEEDBACK_CONFIGS, _RUBRIC_ITEMS

def _tenant_id(client) -> str:
    return str(client.read_project(project_name=PROJECT).tenant_id)

def create_run_rule(client, *, project_name, display_name, filter, sampling_rate=1.0,
                    code_evaluator=None, add_to_annotation_queue_id=None,
                    add_to_dataset_id=None):
    project_id = str(client.read_project(project_name=project_name).id)
    rule = {
        "display_name": display_name,
        "sampling_rate": sampling_rate,
        "session_id": project_id,
        "filter": filter,
        "is_enabled": True,
    }
    if code_evaluator is not None:
        rule["code_evaluator"] = code_evaluator
    if add_to_annotation_queue_id is not None:
        rule["add_to_annotation_queue_id"] = str(add_to_annotation_queue_id)
    if add_to_dataset_id is not None:
        rule["add_to_dataset_id"] = str(add_to_dataset_id)
    resp = client.request_with_retries("POST", "/runs/rules", request_kwargs={"json": rule})
    resp.raise_for_status()
    payload = resp.json()
    tenant = payload.get("tenant_id") or _tenant_id(client)
    return {
        "id": payload["id"],
        "payload": payload,
        "url": f"https://smith.langchain.com/o/{tenant}/projects/p/{project_id}/rules",
    }

_PASS_FAIL = {"type": "categorical",
              "categories": [{"value": 1, "label": "Pass"}, {"value": 0, "label": "Fail"}]}

# The repo's support rubric + the house-style voice label for Align Evals.
NOTEBOOK_FEEDBACK_CONFIGS = [*_FEEDBACK_CONFIGS, ("voice_ready", _PASS_FAIL, None)]
NOTEBOOK_RUBRIC_ITEMS = [
    *_RUBRIC_ITEMS,
    {"feedback_key": "voice_ready",
     "description": ("VOICE replies only: one-sentence acknowledgement, exactly ONE action or outcome, "
                     "ends with a check-in question, and no digit strings, '>' menu symbols, or identifiers."),
     "value_descriptions": {"Pass": "Ready to be spoken", "Fail": "Breaks a house voice rule"},
     "is_required": False},
]

def _find_queue(client, name):
    for q in client.list_annotation_queues():
        if q.name == name:
            return q
    return None

def ensure_queue(client, name, run_ids=None):
    for key, config, lower_better in NOTEBOOK_FEEDBACK_CONFIGS:
        kwargs = {"feedback_config": config}
        if lower_better is not None:
            kwargs["is_lower_score_better"] = lower_better
        try:
            client.create_feedback_config(key, **kwargs)
        except Exception:
            pass  # already exists with this schema
    queue = _find_queue(client, name) or client.create_annotation_queue(
        name=name,
        description="Human review of voice + chat support turns for resolution, privacy, and channel fit.",
        rubric_instructions=(
            "Read the support turn (traces are already redacted). Score whether the issue was "
            "resolved, confirm privacy policy was followed, and check the reply fits its channel. "
            "For VOICE replies, also apply the voice_ready house rule. Flag any raw sensitive value "
            "you can still see in the trace in the notes."
        ),
        rubric_items=NOTEBOOK_RUBRIC_ITEMS,
    )
    if run_ids:
        client.add_runs_to_annotation_queue(queue.id, run_ids=list(run_ids)[:5])
    return queue

queue = ensure_queue(client, scoped("pov-support-iteration-review"))
print(f"Queue: {queue.name} (id={queue.id})")
print(f"{len(NOTEBOOK_RUBRIC_ITEMS)} rubric items attached.")

## Pairwise evaluation (A/B, LLM-as-judge)

Absolute evaluators score each experiment against a rubric. But when you're choosing between two
candidates — a cheaper model vs. the incumbent, a new prompt vs. the old one — the question is
**relative**: *which support reply is better?* Pairwise (A/B) evaluation shows both candidates'
replies to a judge **side by side** and asks it to pick a winner. This sidesteps the calibration
problem where a judge marks everything `True` on an absolute scale but can still reliably say
*A beats B*.

We use `client.evaluate(...)` in **comparative** mode, which takes **two existing experiments**
and a **comparative evaluator**. The evaluator receives the `outputs` from both experiments for
the same example plus the example's inputs and reference outputs, and returns a ranking.

**The decision we're actually making — is a model swap worth it?** The most common A/B a support
team runs is a **model migration**: a newer or cheaper model ships, and before rolling it out to
real callers they need to know whether replies get *better, worse, or indistinguishable* — on
both channels, and without any regression on privacy. So candidate **A** is the support agent on
the **incumbent model** (the current `MODEL`) and candidate **B** is the *same full agent* — same
tools, prompt, playbook skill, and PII middleware — on a **candidate model** (`CANDIDATE_MODEL`).

> **What the judges see:** the agent's reply is produced *after* the Level 1 middleware has
> scrubbed its output, and the experiment is stored through the masking client. Judges therefore
> grade **redacted text** — placeholders like `[REDACTED_EMAIL]` or `****1111` are the privacy
> layer working, not errors.

**In the UI:** open two experiments in *Datasets & Experiments*, select both, and click
*Compare* → *Pairwise* to configure the same judge with no code.

### Candidate A — the support agent on the incumbent model

First run the incumbent: the agent exactly as it ships on the current `MODEL`. We score it with an
inline absolute **`support_quality`** judge — the absolute twin of the pairwise judge below — so
both candidates share the same criteria. A reply is good only if it is **resolved** (grounded,
correct actions), **privacy-safe**, and **fits its channel**. This also gives you each candidate's
absolute `support_quality` alongside the head-to-head win rate.

In [ ]:
from pydantic import BaseModel, Field

class SupportQualityVerdict(BaseModel):
    good: bool = Field(description="True if the turn is resolved, privacy-safe, and fits its channel.")
    justification: str = Field(description="One or two sentences explaining the verdict.")

_SUPPORT_QUALITY_PROMPT = """You grade one turn from a smart-TV customer support agent.

A GOOD turn meets all three bars:
1. RESOLVED - it addresses what the customer actually asked, grounded in real diagnostics or
   published help-center steps, and takes account actions (opt-out, privacy request, ticket)
   only when the customer asked for them.
2. PRIVACY-SAFE - it never discloses viewing history (titles, networks, times watched), never
   reads out contact/payment details or device/network identifiers, never asks for an email,
   phone, SSN, card, or address to verify, and takes NO account action for an unverified caller.
3. FITS THE CHANNEL - VOICE: short, speakable sentences, one step at a time, no markdown, URLs,
   or digit strings. CHAT: concise, scannable steps.

Placeholders like [REDACTED_EMAIL] or ****1111 are expected privacy redactions, not errors.

CHANNEL: {channel}
IDENTITY VERIFIED: {verified}
CUSTOMER MESSAGE:
{message}

ACTIONS THE AGENT TOOK: {actions}

THE AGENT'S REPLY:
{response}

Decide whether the turn is good."""

def support_quality_evaluator(inputs, outputs, reference_outputs=None):
    judge = judge_model.with_structured_output(SupportQualityVerdict)
    v: SupportQualityVerdict = judge.invoke(_SUPPORT_QUALITY_PROMPT.format(
        channel=inputs.get("channel", ""),
        verified="yes" if inputs.get("identity_verified") else "no",
        message=inputs.get("message", ""),
        actions=", ".join(outputs.get("actions", [])) or "none",
        response=outputs.get("response", ""),
    ))
    return {"key": "support_quality", "score": 1.0 if v.good else 0.0, "comment": v.justification}

# Candidate A: the full agent on the incumbent model, scored on support_quality.
results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[support_quality_evaluator],
    experiment_prefix=scoped("support-quality"),
    max_concurrency=2,
)
print(f"Candidate A experiment: {results.experiment_name}")

### Candidate B — the same agent on a candidate model

Now the model we're evaluating for migration. We build the **identical agent** — same system
prompt, playbook skill, and memory from the Context Hub, same tools, same PII middleware — but on
`CANDIDATE_MODEL` instead of the incumbent. Set `CANDIDATE_MODEL` in `.env` to whatever you're
considering (a newer release, a cheaper tier, or a different provider — e.g. `openai:gpt-5-nano`,
`anthropic:claude-haiku-4-5`); it defaults to a sensible alternate so the cell runs out of the
box. With the LLM Gateway on, make sure your workspace has a provider connection for that model.

In [ ]:
# The model we're evaluating for migration. Override in .env; defaults to an
# alternate tier so the cell runs out of the box. If it matches the incumbent
# MODEL, the comparison degenerates to a same-model sanity check (expect ~50%).
INCUMBENT_MODEL = os.environ.get("MODEL") or "(config default)"
CANDIDATE_MODEL = os.environ.get("CANDIDATE_MODEL", "openai:gpt-5-nano")
print(f"Incumbent (A): {INCUMBENT_MODEL}")
print(f"Candidate (B): {CANDIDATE_MODEL}")

# Build the IDENTICAL agent on the candidate model. build_agent() reads the MODEL
# env var via get_model(), so we swap it just for this construction, then restore
# it so the incumbent `agent` (already built in setup) is unaffected.
_saved_model = os.environ.get("MODEL")
os.environ["MODEL"] = CANDIDATE_MODEL
try:
    candidate_agent = build_agent()
finally:
    if _saved_model is None:
        os.environ.pop("MODEL", None)
    else:
        os.environ["MODEL"] = _saved_model

def target_candidate_model(inputs: dict) -> dict:
    return run_support_agent(
        inputs,
        agent_=candidate_agent,
        metadata={"candidate_model": CANDIDATE_MODEL},
        tags=["candidate-model"],
    )

baseline_results = client.evaluate(
    target_candidate_model,
    data=dataset_name,
    evaluators=[support_quality_evaluator],
    experiment_prefix=scoped("support-quality-candidate-model"),
    max_concurrency=2,
)
print(f"Candidate B ({CANDIDATE_MODEL}) experiment: {baseline_results.experiment_name}")

Now the pairwise judge — the **comparative twin of the `support_quality` judge** above. For each
turn it sees the channel, whether the caller was verified, the customer's message, the points a
good resolution `must_address`, the turn's `privacy_expectation`, and **both** candidates' replies
*and the actions each took* (order randomized to avoid position bias). Its priorities, in order:

1. **Privacy is a hard gate.** A reply that discloses viewing history or identifiers, or takes an
   account action for an unverified caller, **loses** — however helpful it is otherwise.
2. **Correct resolution AND correct actions** — the must-address points covered, the right tools
   called, and nothing done that the customer didn't ask for.
3. **Channel fit** — speakable and one-step-at-a-time on voice; concise and scannable on chat.

It has no idea which model produced which reply, so it can't play favorites. The comparative
evaluator returns `scores` positionally for the two runs — `1` for the winner, `0` for the loser
(`0.5`/`0.5` on a tie) — which LangSmith aggregates into the **win rate** that decides the migration.

In [ ]:
import random
from langchain_core.messages import SystemMessage, HumanMessage

class PairwiseGrade(BaseModel):
    """Which reply better handled the customer's turn."""
    preferred: int = Field(description="1 if Reply A is better, 2 if Reply B is better, 0 if tie.")
    reasoning: str = Field(description="One-sentence explanation.")

pairwise_judge_prompt = """You are comparing two replies from a smart-TV customer support agent to the SAME customer turn.

This is the comparative form of the same `support_quality` criteria we judged absolutely above. You'll see the CHANNEL (voice = spoken by text-to-speech, chat = shown in a widget), whether the caller's IDENTITY IS VERIFIED, the CUSTOMER MESSAGE, the points a good resolution MUST ADDRESS, the PRIVACY EXPECTATION for this turn, and for each candidate (A and B) its REPLY and the ACTIONS it took.

Judge in this order:
1. PRIVACY IS A HARD GATE. A reply that discloses viewing history (titles, networks, genres, times watched), reads out contact/payment details or device/network identifiers, asks for an email/phone/SSN/card/address to verify, or takes ANY account action for an unverified caller LOSES - regardless of how helpful it is. If both violate, prefer the less severe violation.
2. RESOLUTION AND ACTIONS. Prefer the reply that covers more of the MUST-ADDRESS points with grounded, accurate steps, AND whose actions are correct: the actions the customer asked for were taken, and nothing was done that they didn't ask for.
3. CHANNEL FIT. On voice, prefer short speakable sentences, one step at a time, no markdown/URLs/digit strings. On chat, prefer concise numbered steps.

Placeholders like [REDACTED_EMAIL] or ****1111 are privacy redactions, not errors.

Return preferred=1 if A is better, preferred=2 if B is better, or preferred=0 only if they are genuinely equivalent. Give one short sentence of reasoning."""

pairwise_judge = judge_model.with_structured_output(PairwiseGrade)

def _candidate_block(label, out):
    actions = ", ".join(out.get("actions", [])) or "none"
    return f"Reply {label} (actions taken: {actions}):\n{out.get('response')}"

def pairwise_evaluator(inputs: dict, outputs: list, reference_outputs: dict) -> dict:
    # `outputs` holds the two experiments' outputs in a fixed order. Shuffle the
    # A/B presentation so the judge can't exploit position, then map back.
    order = [0, 1]
    random.shuffle(order)
    a_idx, b_idx = order
    ref = reference_outputs or {}
    grade: PairwiseGrade = pairwise_judge.invoke([
        SystemMessage(content=pairwise_judge_prompt),
        HumanMessage(content=(
            f"Channel: {inputs.get('channel')}\n"
            f"Identity verified: {'yes' if inputs.get('identity_verified') else 'no'}\n\n"
            f"Customer message: {inputs.get('message')}\n\n"
            f"Must address: {ref.get('must_address', [])}\n\n"
            f"Privacy expectation: {ref.get('privacy_expectation', '')}\n\n"
            f"{_candidate_block('A', outputs[a_idx])}\n\n"
            f"{_candidate_block('B', outputs[b_idx])}"
        )),
    ])
    # Translate the judge's A/B choice back to the original experiment order.
    scores = [0.5, 0.5]
    if grade.preferred == 1:
        scores[a_idx], scores[b_idx] = 1, 0
    elif grade.preferred == 2:
        scores[a_idx], scores[b_idx] = 0, 1
    return {"key": "pairwise_preference", "scores": scores, "comment": grade.reasoning}

### Run the comparison.
`client.evaluate` in comparative mode takes the **two experiment names** (the incumbent A and the
candidate-model B we just ran) and the comparative evaluator. The result view shows per-example
winners and the overall **win rate** — read it alongside each model's absolute `support_quality`,
its cost/latency, and a **per-channel** split (filter the compare view on the `channel` metadata)
to make the call. A candidate that wins on chat but loses on voice isn't ready for the phone line.

In [ ]:
comparison = client.evaluate(
    [results.experiment_name, baseline_results.experiment_name],
    evaluators=[pairwise_evaluator],
    experiment_prefix=scoped("support-pairwise"),
    max_concurrency=2,
)
print("Pairwise comparison complete — open the compare view in LangSmith to see win rates.")

## Align Evals — Tune the Judge to a Human Expert

An LLM-as-judge is only as trustworthy as its agreement with a human expert. For objective checks
— did the reply leak a card number, did it take a forbidden action — the repo already uses
deterministic evaluators, and generic quality is something the model is reasonably calibrated on
out of the box. There's little to tune.

The judgments worth aligning are the **subjective, house-style** ones the model has no prior
about. For our phone line that's **`voice_ready`**: the support team's standard says a VOICE reply
is ready to be spoken only if it does **all four** of:

1. **acknowledges** the issue in one short sentence;
2. gives **exactly one** action or outcome — not a list of steps;
3. **ends with a check-in question** ("Let me know when it's back on?");
4. contains **no digit strings, menu symbols like `>`, or identifiers**.

A reply that is accurate, polite, and even short can still fail — e.g. three steps in a row, or a
menu path written as `Settings > Network`. A model left to its own devices will happily call those
"fine for voice."

**Align Evals** (LangSmith's *Align Evaluator* workflow) closes that gap. You label a handful of
real outputs the way an expert would, then iterate the judge prompt in the **Evaluator Playground**
until its scores match your labels. The metric is the **alignment score** — the % of examples
where the judge agrees with the human. The loop has four steps:

1. **Select runs** — pick real voice replies to label.
2. **Label them** — a human scores each in the annotation queue, then adds it to a reference dataset.
3. **Test the judge prompt** — run it over the labeled set in the Evaluator Playground and read the alignment score.
4. **Refine & repeat** — investigate misaligned examples, tighten the rubric, re-test.

📖 Full guide: [Improve LLM-as-judge evaluators using human feedback →](https://docs.langchain.com/langsmith/improve-judge-evaluator-feedback)

### 1. Select runs — seed the queue with voice replies to label

Align Evals needs real outputs to label, balanced across *voice-ready* (1) and *not* (0). We run a
handful of **voice turns**: some exactly as callers phrase them (the agent usually gives one step
and checks in → likely 1s), and some where the caller **nudges toward a multi-step dump** ("just
tell me all the steps at once") → likely 0s. Then one more `create_run_rule` routes **voice** root
traces into the queue, filtering on the `channel` metadata that `run_turn` stamps on every run.

> The rule keeps routing new voice traces until you disable or delete it — fine for a labeling
> session; turn it off when you have enough examples.

In [ ]:
# A spread of voice turns: plain transcripts (likely voice-ready) mixed with
# callers nudging toward a multi-step answer (likely NOT voice-ready).
align_requests = [
    # Plain voice turns, as the STT engine transcribed them.
    {"case": voice_cases[0], "nudge": ""},
    {"case": voice_cases[1], "nudge": ""},
    {"case": voice_cases[3], "nudge": ""},
    # Nudged toward a dump of steps, menu paths, and details.
    {"case": voice_cases[1], "nudge": "just tell me all the steps at once I don't want to go back and forth"},
    {"case": voice_cases[3], "nudge": "read me the whole fix list and the exact menu path and the ticket number"},
    {"case": voice_cases[4], "nudge": "give me everything I need to do for both things right now in one go"},
]

for r in align_requests:
    turn = dict(r["case"])
    if r["nudge"]:
        turn["message"] = f"{turn['message']} {r['nudge']}"
    out = run_support_agent(turn, metadata={"align": "voice_ready"}, tags=["align"])
    preview = (out["response"] or "")[:110].replace(chr(10), " ")
    print(f"[{'nudged' if r['nudge'] else 'plain '}] {preview}...")
print(f"Generated {len(align_requests)} voice runs.")

# Route voice root traces into the review queue for labeling.
align_seed_rule = create_run_rule(
    client,
    project_name=PROJECT,
    display_name=scoped("pov-align-voice-ready"),
    sampling_rate=1.0,
    filter='and(eq(is_root, true), and(eq(metadata_key, "channel"), eq(metadata_value, "voice")))',
    add_to_annotation_queue_id=queue.id,
)
print("Seeded queue for labeling:", align_seed_rule["url"])

### 2. Label the runs (in the annotation queue)

Open the review queue and score **`voice_ready`** on each voice reply the way the **support-team
expert** would — not the way a lenient model would:

- Score **1** only if the reply does **all four**: a one-sentence acknowledgement, **exactly one**
  action or outcome, a closing **check-in question**, and **no** digit strings, `>` menu symbols,
  or identifiers.
- Score **0** if any of the four is missing — even a correct, friendly reply that lists three steps
  or writes `Settings > Network > Reset` is not voice-ready by house style.

The traces you're labeling were redacted before they reached LangSmith, so labelers never see raw
customer data. A placeholder such as `[REDACTED_US_PHONE]` in a reply still counts as an
identifier slot the agent should not have spoken — score it 0.

After scoring an example, click **Add to Reference Dataset**. Aim for **~20 examples** with a
**balance of 0s and 1s** (run the seed cell a few times if you need more). Diversity matters more
than volume.

🔗 Queue (review UI): **Annotation Queues** in the LangSmith sidebar → `pov-support-iteration-review-<you>`.

### 3. Test the judge prompt in the Evaluator Playground

Create the aligned evaluator from your labeled data:

1. **Datasets & Experiments** → your reference dataset → **+ Evaluator** → **Create from labeled data**.
2. Name the feedback key **`voice_ready`**.
3. In the **Evaluator Playground**, paste the starter judge prompt below, enable **reasoning**, and click **Start Alignment**.

LangSmith runs the judge over your labeled set and shows the **alignment score** (% agreement with
your labels), plus which examples it got wrong.

**Starter `voice_ready` judge prompt** (paste into the playground):

```text
You are grading a reply from a smart-TV support agent's PHONE LINE. The reply will be
read aloud by text-to-speech to a caller.

House-style rule: a reply is VOICE-READY (score 1) only if it does ALL FOUR of:
  1. Acknowledge - opens by acknowledging the caller's issue in ONE short sentence.
  2. One thing - gives exactly ONE action for the caller to take, or ONE outcome
     (e.g. "I've turned that off"). Two or more steps, or a list, fails.
  3. Check-in - ends with a question that checks in with the caller
     (e.g. "Let me know when it's back on?").
  4. Speakable - contains no digit strings (numbers must be spoken as words), no menu
     symbols such as ">", no markdown or URLs, and no identifiers or redaction
     placeholders (anything like [REDACTED_...] or ****1234).

If ANY of the four is missing, the reply is NOT voice-ready (score 0) - even if it is
accurate, polite, and short.

Return score=1 (voice-ready) or score=0 (not voice-ready) and one sentence naming which
of the four rules pass and which fail.
```

You'll likely see the judge start too lenient — passing replies that bundle two steps or write a
menu path with `>` — which is exactly the gap alignment closes.

### 4. Refine & repeat until the judge agrees with you

Iterate in the playground; save the prompt whenever the alignment score improves (updates aren't
saved automatically — the playground compares against your last *saved* version).

Tactics that move the score:

- **Investigate misaligned examples** — hover the LLM score to read its **reasoning**; group the
  misses into failure modes (e.g. "counts 'restart the router and the TV' as one action") and add an
  explicit instruction for each.
- **Pin down the house rule** — spell out edge cases the model gets wrong (e.g. "a ticket ID spoken
  character by character in words is fine; 'CS-48213' is a digit string and fails", or "a closing
  'Is there anything else?' counts as a check-in only after an outcome, not after an action").
- **Add more labels, then re-check** — expand the reference set so you don't overfit the first handful.

When the alignment score is high enough to trust, promote this `voice_ready` judge into an
**online evaluator**: on the tracing project, *Rules → + New Rule → LLM-as-judge*, paste the aligned
prompt, and filter it to voice traffic with the same expression as the seeding rule
(`and(eq(is_root, true), and(eq(metadata_key, "channel"), eq(metadata_value, "voice")))`). It runs
on traces that were already masked before egress, and it complements the repo's deterministic
`channel-format` code evaluator (`src/pov_agent/langsmith/online_eval.py`): that one catches digit
runs and markdown cheaply on every reply; `voice_ready` adds the conversational shape your support
team actually cares about — aligned to their standard, scoring every new voice turn automatically.